In [7]:
import pandas as pd
import random

In [8]:
print('A')

A


In [39]:
guesses_path = "../data/valid_guesses.csv"
file_guesses = pd.read_csv(guesses_path)
answers_path = "../data/valid_answers.csv"
file_answers = pd.read_csv(answers_path)

valid_guesses = file_guesses['word']
valid_answers = file_answers['word']


In [10]:
file_guesses

,word
0,ababa
1,abaca
2,abaco
3,abada
4,abade
...,...
9142,zurpa
9143,zurra
9144,zurre
9145,zurro


In [11]:
file_answers

,word
0,abano
1,abono
2,abril
3,abrir
4,abuso
...,...
1437,zerar
1438,zinco
1439,ziper
1440,zonas


## Criação do ambiente do jogo:

In [12]:
def try_word_check(try_word, target_word):
    """
    1  -> letra correta no lugar correto
    0  -> letra existe mas em outra posição
    -1 -> letra não existe
    """

    result = [-1] * len(try_word)

    available = list(target_word)

    # Marca os acertos exatos (1)
    for i in range(len(try_word)):
        if try_word[i] == target_word[i]:
            result[i] = 1
            available[i] = None  # consome a letra

    # Marca letras existentes fora de posição (0)
    for i in range(len(try_word)):
        if result[i] == -1:
            if try_word[i] in available:
                result[i] = 0

                # consome apenas UMA ocorrência da letra
                letter_index = available.index(try_word[i])
                available[letter_index] = None
                
    return result

In [13]:
def run():
    target_word = random_word_generate()
    tries = 0
    while(tries<5):
        try_word = input()
        result = try_word_check(try_word, target_word)
        if result == True:
            return True
        tries+=1
    print(f"The word has {target_word}.")
    return False

In [14]:
class Termoo:
    def run_game():
        target_word = random_word_generate()
        tries = 0
        while(tries<5):
            try_word = input()
            result = try_word_check(try_word, target_word)
            if result == True:
                return True
            tries+=1
        print(f"The word has {target_word}.")
        return False

## Helper functions

In [15]:
def find_the_best_first_word(valid_answers, valid_guesses):
    words = {}
    
    for guess in valid_guesses:
        words[guess] = 0 
        for answer in valid_answers:
            score = sum(try_word_check(guess, answer))
            words[guess] += score 

    return words

In [16]:
words_sum = find_the_best_first_word(file_answers['word'], file_guesses['word'])

In [17]:
words_sum_ordenado_decrescente = dict(sorted(words_sum.items(), key=lambda item: item[1], reverse=True))

In [18]:
# Melhores tentativas
list(words_sum_ordenado_decrescente.items())[:10]

[('serao', -2733),
 ('terao', -2801),
 ('lerao', -2849),
 ('sarao', -2922),
 ('tirao', -2956),
 ('relao', -2957),
 ('cario', -2974),
 ('renao', -2980),
 ('carao', -2986),
 ('risao', -2995)]

In [19]:
# Piores tentativas
list(words_sum_ordenado_decrescente.items())[-10:]

[('zungu', -6141),
 ('husky', -6158),
 ('quiui', -6168),
 ('equeu', -6169),
 ('inibi', -6170),
 ('jiqui', -6223),
 ('juqui', -6243),
 ('ubucu', -6290),
 ('buggy', -6317),
 ('ucubu', -6342)]

## Stochastic Agents:

In [20]:
def test_agents(agent, target_word, epochs=10000):
    tries_keys = ['first', 'second', 'third', 'fourth', 'fifth', 'lose']
    tries_count = [0] * 6
      # o somatorio de cada item deve ser = epochs
      # sum(key.items) = epochs
      # first_try = 1try / epochs (probabilidade de acertar em cada tentativa)

    # tries:
    # 0 -> acertou de primeira
    # 1 -> acertou na segunda tentativa
    # 2 -> acertou na terceira tentativa
    # 3 -> acertou na quarta tentativa
    # 4 -> acertou na quinta tentativa
    # 5 -> perdeu
    # ...

    # Um agente de fato deveria ter essas entradas:
    # Uma lista de dicionario chamada history
    # history: [
    #     { 'abc' : [1, 0, -1] }
    #     { 'afb' : [1, -1, 1] }
    # ]
    # TARGET_WORD NÃO DEVERIA ENTRAR DENTRO DO AGENTE
    # O return de um agente deveria ser apenas a palavra e nada mais
    # return: 'aeb'
    
    for epoch in range(epochs):
        result = agent(target_word)
        if result == -1: # significa que ele perdeu
            tries_count[5]+=1 # lose aumenta
        else:
            tries_count[result]+=1
    
    return tries_count
    #return dict(zip(tries_keys, tries_count))

In [21]:
# ------------------------------------------
#                Bogo Agent
# ------------------------------------------

def bogo_agent(*args):
    """
    Bogo Agent tenta acertar as palavras de maneira aleatória.
    epochs=10000000 -> [7020, 7029, 6988, 6990, 6826, 9965147]
    """
    return random.choice(valid_answers)

In [22]:
# ------------------------------------------
#            Simple Reflex Agent
# ------------------------------------------

def try_word_check(try_word, target_word):
    """
    1  -> letra correta no lugar correto
    0  -> letra existe mas em outra posição
    -1 -> letra não existe
    """

    result = [-1] * len(try_word)

    available = list(target_word)

    # Marca os acertos exatos (1)
    for i in range(len(try_word)):
        if try_word[i] == target_word[i]:
            result[i] = 1
            available[i] = None  # consome a letra

    # Marca letras existentes fora de posição (0)
    for i in range(len(try_word)):
        if result[i] == -1:
            if try_word[i] in available:
                result[i] = 0

                # consome apenas UMA ocorrência da letra
                letter_index = available.index(try_word[i])
                available[letter_index] = None
                
    return result

def ranking_words(valid_answers, valid_guesses):
    words = {}
    for guess in valid_guesses:
        words[guess] = 0 
        for answer in valid_answers:
            score = sum(try_word_check(guess, answer))
            words[guess] += score 
    return words

def simple_reflex_agent(history, valid_answers=file_answers['word'], valid_guesses=file_guesses['word']):
    """
    Simple Reflex Agent faz tentativas com a vencedora da funcao:
    ranking_words()
    """

    if(not history):
        #ranked_words = ranking_words(valid_answers, valid_guesses)
        #return max(ranked_words, key=ranked_words.get)
        return 'serao'

    # History é uma LISTA de DICIONÁRIOS
    # history = [
    #     {'abc': [1, 1, -1]},
    #     {'dec': [-1, -1, -1]}
    # ]

    # attempt -> {'abc': [1, 1, -1]}
    # word -> 'abc'
    # score -> [1, 1, -1]

    letters_in_word = set()
    for attempt in history:
        for word, score in attempt.items():
            for i in range(len(word)):
                if score[i] >= 0:  # Se for 0 ou 1, a letra existe
                    letters_in_word.add(word[i])
                    
    wrong_letters = set()
    for attempt in history:
        for word, score in attempt.items():
            for i in range(len(word)):
                if score[i] == -1 and word[i] not in letters_in_word:
                    wrong_letters.add(word[i])
    
    # wrong_letters = ['c', 'd', 'e', 'c']

    # Remover todas as palavras que contem as wrong_letters
    valid_answers = [
        word for word in valid_answers 
        if not any(letter in word for letter in wrong_letters)
    ]

    history_words = set()
    for attempt in history:
        history_words.update(attempt.keys())

    valid_answers = [
        word for word in valid_answers 
        if word not in history_words
    ]

    # valid_guesses = [
    #     word for word in valid_guesses 
    #     if not any(letter in word for letter in wrong_letters)
    # ]

    # Filtro de letras nas posições corretas ou quase corretas

    for attempt in history:
        for word, score in attempt.items():
            for i in range(len(word)):
                letter = word[i]
                s = score[i]

                if s == 1: # Verde: a letra tem que estar nesta posição
                    valid_answers = [w for w in valid_answers if w[i] == letter]

                if s == 0: # Amarelo: a letra tem que existir mas não nesta posição
                    valid_answers = [w for w in valid_answers if letter in w and w[i] != letter]


    # Rankeamento das palavras
    ranked_words = ranking_words(valid_answers, valid_answers)
    #print(ranked_words)
    return max(ranked_words, key=ranked_words.get)

In [23]:
target_word = bogo_agent()
history = []
target_word

'amplo'

In [24]:
guess = simple_reflex_agent(history, valid_answers, valid_guesses)
history.append({guess: try_word_check(guess, target_word)})
guess

'serao'

In [25]:
guess = simple_reflex_agent(history, valid_answers, valid_guesses)
guess

'canto'

In [26]:
history = [
    {'serao': [-1, -1, 0, -1, 0]},
    {'facao': [-1, -1, -1, -1, 0]},
    {'facao': [-1, -1, -1, -1, 0]},
    {'rumor': [1, 1, -1, 1, 1]},
]

In [27]:
target_word = random.choice(valid_answers)
history = []
tries = -1
agent = simple_reflex_agent

for attempt in range(5): # O agente tem 5 tentativas
    try_word = agent(history, valid_answers, valid_guesses)
    history.append({try_word: try_word_check(try_word, target_word)})
    print(f"{attempt + 1}º Run {history}\n\n")
    if(try_word == target_word):
        tries = attempt
        break

1º Run [{'serao': [-1, -1, -1, 0, -1]}]


2º Run [{'serao': [-1, -1, -1, 0, -1]}, {'mania': [-1, -1, -1, 0, 1]}]


3º Run [{'serao': [-1, -1, -1, 0, -1]}, {'mania': [-1, -1, -1, 0, 1]}, {'caixa': [-1, -1, 0, -1, 1]}]


4º Run [{'serao': [-1, -1, -1, 0, -1]}, {'mania': [-1, -1, -1, 0, 1]}, {'caixa': [-1, -1, 0, -1, 1]}, {'pilha': [-1, 1, 1, 1, 1]}]


5º Run [{'serao': [-1, -1, -1, 0, -1]}, {'mania': [-1, -1, -1, 0, 1]}, {'caixa': [-1, -1, 0, -1, 1]}, {'pilha': [-1, 1, 1, 1, 1]}, {'filha': [1, 1, 1, 1, 1]}]




In [28]:
tries, target_word, history

(4,
 'filha',
 [{'serao': [-1, -1, -1, 0, -1]},
  {'mania': [-1, -1, -1, 0, 1]},
  {'caixa': [-1, -1, 0, -1, 1]},
  {'pilha': [-1, 1, 1, 1, 1]},
  {'filha': [1, 1, 1, 1, 1]}])

In [29]:
history

[{'serao': [-1, -1, -1, 0, -1]},
 {'mania': [-1, -1, -1, 0, 1]},
 {'caixa': [-1, -1, 0, -1, 1]},
 {'pilha': [-1, 1, 1, 1, 1]},
 {'filha': [1, 1, 1, 1, 1]}]

In [30]:
teste = ['teste', 'gorda']
teste[1]

history = [
    {'abc': [1, -1, -1]},
    {'dec': [-1, -1, -1]}
]

wrong_letters = []
for attempt in history:
    for word, score  in attempt.items():
        for i in range(len(word)):
            if(score[i] == -1):
                wrong_letters.append(word[i])

wrong_letters

pattern = '|'.join(wrong_letters)
pattern

'b|c|d|e|c'

In [31]:
valid_answers

[
    word for word in valid_answers 
    if not any(letter in word for letter in wrong_letters)
]

['afago',
 'afins',
 'agora',
 'aguia',
 'algoz',
 'algum',
 'aliar',
 'alias',
 'almas',
 'altar',
 'altos',
 'aluno',
 'amago',
 'amora',
 'ampla',
 'amplo',
 'angra',
 'animo',
 'anjos',
 'ansia',
 'antro',
 'anual',
 'anzol',
 'apito',
 'apoio',
 'armar',
 'aroma',
 'arroz',
 'aspas',
 'assar',
 'assim',
 'astro',
 'atlas',
 'atomo',
 'atras',
 'atriz',
 'atroz',
 'atual',
 'atuar',
 'aulas',
 'autor',
 'autos',
 'avaro',
 'aviao',
 'aviso',
 'axila',
 'faixa',
 'falar',
 'falir',
 'falsa',
 'falso',
 'farao',
 'farol',
 'farpa',
 'farra',
 'farsa',
 'farto',
 'fatal',
 'fator',
 'fatos',
 'fauna',
 'favor',
 'fiapo',
 'filha',
 'filho',
 'final',
 'finos',
 'firma',
 'fixar',
 'fixos',
 'flora',
 'fluir',
 'fluor',
 'fluxo',
 'fofas',
 'fogao',
 'fogos',
 'folga',
 'folha',
 'folia',
 'forma',
 'forno',
 'forum',
 'fossa',
 'fosso',
 'fotos',
 'frias',
 'frios',
 'frota',
 'fruta',
 'fruto',
 'fugaz',
 'fugir',
 'fumar',
 'fungo',
 'funil',
 'furar',
 'furia',
 'furor',
 'furos',


In [ ]:
def simple_reflex_agent(target_word):
    """
    Simple Reflex Agent faz tentativas usando a maior soma da função
    find_the_best_first_word()

    epochs=10000000 -> [5, 123, 2415, 215125, 521525, 9965147]

    O retorno deve ser um número entre 1~5 ou -1
    """
    history = []
    if(not history):
        return find_the_best_first_word(valid_answers, valid_guesses)

    # 'serao' -> sum = 10000
    tries = 0
    for answer in valid_answers:
        
        if(try_word == target_word):
            return tries
      
        
        letters_result = try_word_check(try_word, target_word)
        # [1, 0, 0, 1, -1]


        invalid_letters = [try_word[i] for i in range(len(try_word)) if letters_result[i] == -1]
        # letras -1 serão salva em invalid letters

        
        
        valid_answers = valid_answers[valid_answers != try_word]
        # 'serao' removida
        
        tries+=1
        
    return -1
    
        score = sum(try_word_check(guess, answer))
        words[guess] += score 


In [33]:
try_word = "aexxz"
letters_result = try_word_check(try_word, "abcde")
# [1, 0, 0, 1, -1]
print(letters_result)


invalid_letters = [try_word[i] for i in range(len(try_word)) if letters_result[i] == -1]
invalid_letters

[1, 0, -1, -1, -1]


['x', 'x', 'z']

In [34]:
history = []
if(not history):
    print('A')

A


In [ ]:
epochs = 1000000
test_results = test_agents(bogo_agent, valid_answers, epochs)
test_results

In [ ]:
sum(test_results)

In [ ]:
# Probabilidade:
# Bogo Agent é extremamente ruim, quase 100% das vezes ele vai errar
[x / epochs for x in test_results]

In [ ]:
'2' + '2'

In [ ]:
def test_agents(agent, epochs=10):
    tries_keys = ['first', 'second', 'third', 'fourth', 'fifth', 'lose']
    tries_count = [0] * 6
    
    for epoch in range(epochs):
        
        target_word = random.choice(valid_answers)
        history, tries = [], -1
        
        for attempt in range(5): # O agente tem 5 tentativas
            try_word = agent(history, valid_answers, valid_guesses)
            history.append({try_word: try_word_check(try_word, target_word)})
            #print(f"{attempt + 1}º Run {history}\n\n")
            if(try_word == target_word):
                tries = attempt
                break
        
        tries_count[tries] += 1 
    
    return tries_count
    #return dict(zip(tries_keys, tries_count))

In [ ]:
test_agents(simple_reflex_agent, 100000)

In [ ]:
test_agents(bogo_agent, 100000)

In [ ]:
sum([74, 52, 63, 54, 55])/1000

In [ ]:
sum([0, 1079, 4001, 3372, 1166])/1000

In [ ]:
history = [list() for _ in range(1)]
history